# Stats

Scrape team data from [Basketball Stats Vlaanderen](https://app.basketballstatsvlaanderen.be).

## 1. Get all teams for a competition

The competition page (e.g. https://app.basketballstatsvlaanderen.be/competitions/BVBL26279130OVHSE31A) embeds the data it fetched server-side as a JSON blob inside an inline `<script>` tag (`rankedTeams`), so a plain `requests` GET is enough — no headless browser needed.

In [2]:
import requests
import re
import json
import pandas as pd

BASE_URL = "https://app.basketballstatsvlaanderen.be"


def get_competition_teams(competition_guid):
    """Fetch a competition page and return (competition_info, teams_df)."""
    url = f"{BASE_URL}/competitions/{competition_guid}"
    response = requests.get(url, timeout=20)
    response.raise_for_status()
    html = response.text

    competition_info = None
    teams = None

    for script in re.findall(r"<script[^>]*>(.*?)</script>", html, re.DOTALL):
        if "rankedTeams" not in script:
            continue
        payload = json.loads(script)
        body = json.loads(payload["body"])
        competition_info = body.get("competition")
        teams = body.get("rankedTeams")
        break

    if teams is None:
        raise ValueError("Could not find 'rankedTeams' data on the competition page")

    teams_df = pd.DataFrame(teams)
    return competition_info, teams_df

In [3]:
COMPETITION_GUID = "BVBL26279130OVHSE31A"

competition_info, teams_df = get_competition_teams(COMPETITION_GUID)

print(competition_info)
teams_df

{'id': 10397889, 'Guid': 'BVBL26279130OVHSE31A', 'Name': '3e Prov. Heren Oost-Vlaanderen A', 'RegionId': 'BVBL9130', 'Category': 'Heren Senioren', 'createdAt': '2026-07-15T12:35:48.343Z', 'updatedAt': '2026-07-15T12:35:48.343Z'}


,Guid,Name,Ranking,PointsAgainst,PointsMade,GameCount,GamesEqual,GamesForfait,GamePoints,GamesLost,GamesWon
0,BVBL1061HSE 2,BBC Helios SanoRice Zottegem HSE B,-,0,0,0,0,0,0,0,0
1,BVBL1037HSE 2,BBC Haantjes Certifisc Oudenaarde HSE B,-,0,0,0,0,0,0,0,0
2,BVBL1168HSE 1,KBBC Wasocub Waasmunster vzw HSE A,-,0,0,0,0,0,0,0,0
3,BVBL1054HSE 1,BBC Lokeren HSE A,-,0,0,0,0,0,0,0,0
4,BVBL1226HSE 4,Gent-Oost Eagles HSE D,-,0,0,0,0,0,0,0,0
5,BVBL1289HSE 1,BBC Wildcats Gavere HSE A,-,0,0,0,0,0,0,0,0
6,BVBL1460HSE 1,Erembodegem Silverbacks HSE A,-,0,0,0,0,0,0,0,0
7,BVBL1404HSE 1,BBC Hotshots Destelbergen HSE A,-,0,0,0,0,0,0,0,0
8,BVBL1184HSE 1,KBBC Eksaarde HSE A,-,0,0,0,0,0,0,0,0
9,BVBL1029HSE 3,Koninklijke Sint-Niklase Condors HSE C,-,0,0,0,0,0,0,0,0


In [4]:
# Keep just the team identity columns for now
teams_df[["Guid", "Name"]]

,Guid,Name
0,BVBL1061HSE 2,BBC Helios SanoRice Zottegem HSE B
1,BVBL1037HSE 2,BBC Haantjes Certifisc Oudenaarde HSE B
2,BVBL1168HSE 1,KBBC Wasocub Waasmunster vzw HSE A
3,BVBL1054HSE 1,BBC Lokeren HSE A
4,BVBL1226HSE 4,Gent-Oost Eagles HSE D
5,BVBL1289HSE 1,BBC Wildcats Gavere HSE A
6,BVBL1460HSE 1,Erembodegem Silverbacks HSE A
7,BVBL1404HSE 1,BBC Hotshots Destelbergen HSE A
8,BVBL1184HSE 1,KBBC Eksaarde HSE A
9,BVBL1029HSE 3,Koninklijke Sint-Niklase Condors HSE C


## 1b. Get all clubs/teams for a competition with no ranking data yet (e.g. a cup round)

Some competitions — typically early cup rounds like `BVBL24259130BOVHSEVR` ("Beker van
Oost-Vlaanderen Heren Voorronde", see https://vblstats.wisseq.eu/Reeks/BVBL24259130BOVHSEVR) —
have an empty `rankedTeams` list because no standings have been computed. The competition page
still embeds the round's match schedule as a separate JSON blob (same page, same inline-`<script>`
mechanism as `get_competition_teams`), so the participating clubs/teams and their `TeamGuid`s can
be recovered from the unique `HomeTeam`/`AwayTeam` pairs in that schedule instead.

In [8]:
def get_competition_teams_from_matches(competition_guid):
    """Fetch a competition page and return (competition_info, teams_df) built from the
    embedded match schedule instead of rankedTeams (useful when rankedTeams is empty,
    e.g. for cup rounds that don't have standings)."""
    url = f"{BASE_URL}/competitions/{competition_guid}"
    response = requests.get(url, timeout=20)
    response.raise_for_status()
    html = response.text

    competition_info = None
    matches = None

    for script in re.findall(r"<script[^>]*>(.*?)</script>", html, re.DOTALL):
        try:
            payload = json.loads(script)
            body = json.loads(payload["body"])
        except (json.JSONDecodeError, KeyError, TypeError):
            continue

        if isinstance(body, dict) and "competition" in body:
            competition_info = body["competition"]
        elif isinstance(body, list) and body and "HomeTeam" in body[0]:
            matches = body

    if matches is None:
        raise ValueError("Could not find match data on the competition page")

    teams = {}
    for m in matches:
        teams[m["HomeTeamGuid"]] = m["HomeTeam"]
        teams[m["AwayTeamGuid"]] = m["AwayTeam"]

    teams_df = (
        pd.DataFrame(sorted(teams.items()), columns=["Guid", "Name"])
        .sort_values("Name")
        .reset_index(drop=True)
    )
    return competition_info, teams_df


CUP_COMPETITION_GUID = "BVBL24259130BOVHSEVR"
CUP_COMPETITION_GUID = "BVBL25269130OVHSE31A"
CUP_COMPETITION_GUID2 = "BVBL25269130OVHSE31B"

cup_competition_info, cup_teams_df = get_competition_teams_from_matches(CUP_COMPETITION_GUID)
cup_competition_info2, cup_teams_df2 = get_competition_teams_from_matches(CUP_COMPETITION_GUID2)

cup_competition_info

ALLTEAMS = pd.concat([cup_teams_df, cup_teams_df2], ignore_index=True)
ALLTEAMS 

,Guid,Name
0,BVBL1068HSE 4,ABO LDP Donza HSE D
1,BVBL1253HSE 1,BBC Assenede HSE A
2,BVBL1136HSE 4,BBC Falco Gent HSE D
3,BVBL1404HSE 1,BBC Hotshots Destelbergen HSE A
4,BVBL1343HSE 1,Basket Meetjesland HSE A
5,BVBL1161HSE 3,KBBC Bavi Gent HSE C
6,BVBL1184HSE 1,KBBC Eksaarde HSE A
7,BVBL1142HSE 2,KBBC Sparta Laarne HSE B
8,BVBL1168HSE 1,KBBC Wasocub Waasmunster vzw HSE A
9,BVBL1047HSE 2,Koninklijke BBC Scheldejeugd Temse HSE B


## 2. Get a club/team's season matches

A club-team page (e.g. https://app.basketballstatsvlaanderen.be/clubs/BVBL1037/BVBL1037HSE%20%202?season=2526) embeds several JSON blobs the same way: the team's match schedule/results for the selected season, the team info, the club info (with its other teams), and the list of seasons available to switch between.

In [9]:
def get_club_team_season(club_guid, team_guid, season):
    """Fetch a club/team season page and return (team_info, club_info, matches_df, seasons)."""
    url = f"{BASE_URL}/clubs/{club_guid}/{team_guid}"
    response = requests.get(url, params={"season": season}, timeout=20)
    response.raise_for_status()
    html = response.text

    team_info = None
    club_info = None
    matches = None
    seasons = None

    for script in re.findall(r"<script[^>]*>(.*?)</script>", html, re.DOTALL):
        try:
            payload = json.loads(script)
            body = json.loads(payload["body"])
        except (json.JSONDecodeError, KeyError, TypeError):
            continue

        if isinstance(body, dict) and "team" in body:
            team_info = body["team"]
        elif isinstance(body, dict) and "club" in body:
            club_info = body["club"]
        elif isinstance(body, list) and body and "HomeTeam" in body[0]:
            matches = body
        elif isinstance(body, list) and body and isinstance(body[0], str):
            seasons = body

    if matches is None:
        raise ValueError("Could not find match data on the club/team page")

    matches_df = pd.DataFrame(matches)
    return team_info, club_info, matches_df, seasons

In [4]:
CLUB_GUID = "BVBL1037"
TEAM_GUID = "BVBL1037HSE  2"
SEASON = "2526"

team_info, club_info, matches_df, seasons = get_club_team_season(CLUB_GUID, TEAM_GUID, SEASON)

print(team_info)
print("available seasons:", seasons)
matches_df

{'id': 4531337, 'Guid': 'BVBL1037HSE  2', 'ClubGuid': 'BVBL1037', 'Name': 'BBC Haantjes Certifisc Oudenaarde HSE B', 'Category': 'Heren Senioren', 'Competitions': [{'key': '6BVBL26279130OVHSE31A', 'guid': 'BVBL26279130OVHSE31A', 'naam': '3e Prov. Heren Oost-Vlaanderen A', 'sort': None, 'type': 6, 'catId': None, 'teams': [], 'wfInfo': None, 'categorie': None, 'regioGUID': None, 'regioNaam': None, 'relatedPoules': None}, {'key': '6BVBL26279130BOVHSEVR', 'guid': 'BVBL26279130BOVHSEVR', 'naam': 'Beker van Oost-Vlaanderen Heren Voorronde', 'sort': None, 'type': 6, 'catId': None, 'teams': [], 'wfInfo': None, 'categorie': None, 'regioGUID': None, 'regioNaam': None, 'relatedPoules': None}], 'IsStatsPublic': False, 'IsExtraStatsPublic': False, 'createdAt': '2024-08-02T21:16:47.831Z', 'updatedAt': '2024-08-02T21:16:47.831Z'}
available seasons: ['2526', '2425']


,Date,Guid,Time,Result,AwayTeam,HomeTeam,AwayTeamGuid,HomeTeamGuid
0,2027-04-17,BVBL26279130OVHSE31ABN,20:15,,ABO LDP Donza HSE D,BBC Haantjes Certifisc Oudenaarde HSE B,BVBL1068HSE 4,BVBL1037HSE 2
1,2027-04-11,BVBL26279130OVHSE31AKB,17:00,,BBC Haantjes Certifisc Oudenaarde HSE B,KBBC Sparta Laarne HSE C,BVBL1037HSE 2,BVBL1142HSE 3
2,2027-03-21,BVBL26279130OVHSE31AFB,17:00,,BBC Haantjes Certifisc Oudenaarde HSE B,BBC Wildcats Gavere HSE A,BVBL1037HSE 2,BVBL1289HSE 1
3,2027-03-13,BVBL26279130OVHSE31AMB,18:20,,BBC Haantjes Certifisc Oudenaarde HSE B,BBC Feniks Futuria Gent HSE B,BVBL1037HSE 2,BVBL1446HSE 2
4,2027-03-06,BVBL26279130OVHSE31ABL,20:15,,Amon Jeugd Gentson HSE D,BBC Haantjes Certifisc Oudenaarde HSE B,BVBL1414HSE 4,BVBL1037HSE 2
5,2027-02-28,BVBL26279130OVHSE31ACB,17:00,,BBC Haantjes Certifisc Oudenaarde HSE B,KBBC Wasocub Waasmunster vzw HSE A,BVBL1037HSE 2,BVBL1168HSE 1
6,2027-02-20,BVBL26279130OVHSE31ABH,20:15,,BBC Hotshots Destelbergen HSE A,BBC Haantjes Certifisc Oudenaarde HSE B,BVBL1404HSE 1,BVBL1037HSE 2
7,2027-02-06,BVBL26279130OVHSE31ABA,20:15,,BBC Helios SanoRice Zottegem HSE B,BBC Haantjes Certifisc Oudenaarde HSE B,BVBL1061HSE 2,BVBL1037HSE 2
8,2027-01-30,BVBL26279130OVHSE31AGB,20:30,,BBC Haantjes Certifisc Oudenaarde HSE B,Erembodegem Silverbacks HSE A,BVBL1037HSE 2,BVBL1460HSE 1
9,2027-01-23,BVBL26279130OVHSE31ABI,20:15,,KBBC Eksaarde HSE A,BBC Haantjes Certifisc Oudenaarde HSE B,BVBL1184HSE 1,BVBL1037HSE 2


## 3. Player stats

**Note on the competition-wide player-stats pages** (e.g. https://app.basketballstatsvlaanderen.be/competitions/BVBL25269130OVHSE31A/player-stats): the page itself has no embedded data — it calls `GET /api/competitions/{guid}/playerStats` client-side, which returns **401 Unauthorized** without a logged-in ("Stats Plus") account. There's no public way to page through that endpoint, so it's skipped here.

What **is** public is the per-player endpoint behind an individual player page (e.g. https://app.basketballstatsvlaanderen.be/players/BVBL661847?season=2526), which calls `GET /api/players/{guid}?season=...` and returns 200 without login: the player's per-game box score stats for that season, plus which team (`GameTeam`) they played for in each game.

In [5]:
def get_player(player_guid, season="2526"):
    """Fetch a player's per-game stats for a season, plus a compact team/metadata table.

    Returns (player_name, games_df, playerdetails_df) where playerdetails_df has one row
    per player with a 'playerdetailsTable' column compactly listing the team(s) the player
    belongs to this season (a player can appear for more than one team, e.g. after a transfer).
    """
    url = f"{BASE_URL}/api/players/{player_guid}"
    response = requests.get(url, params={"season": season}, timeout=20)
    response.raise_for_status()
    data = response.json()

    player_name = data.get("playerName")
    games = data.get("games", [])

    game_rows = []
    for g in games:
        game_team = g.get("GameTeam") or {}
        game = game_team.get("Game") or {}
        game_rows.append({
            "playerGuid": player_guid,
            "playerName": player_name,
            "gameGuid": game.get("Guid"),
            "gameDate": game.get("Date"),
            "teamGuid": game_team.get("Guid"),
            "teamName": game_team.get("Name"),
            "number": g.get("Number"),
            "functionLetter": g.get("FunctionLetter"),
            "starter": g.get("Starter"),
            "totalMinutes": g.get("TotalMinutes"),
            "normalizedMinutes": g.get("NormalizedMinutes"),
            "freeThrows": g.get("FreeThrows"),
            "fieldGoals": g.get("FieldGoals"),
            "threePointers": g.get("ThreePointers"),
            "totalScore": g.get("TotalScore"),
            "plusMinus": g.get("PlusMinus"),
            "faults": g.get("Faults"),
        })

    games_df = pd.DataFrame(game_rows)

    if not games_df.empty:
        teams = (
            games_df[["teamGuid", "teamName"]]
            .drop_duplicates()
            .dropna()
        )
        teams_compact = "; ".join(f"{row.teamName} ({row.teamGuid.strip()})" for row in teams.itertuples())
    else:
        teams_compact = None

    playerdetails_df = pd.DataFrame([{
        "playerGuid": player_guid,
        "playerName": player_name,
        "season": season,
        "playerdetailsTable": teams_compact,
    }])

    return player_name, games_df, playerdetails_df

In [16]:
PLAYER_GUID = "BVBL661847"
SEASON = "2526"

player_name, player_games_df, playerdetails_df = get_player(PLAYER_GUID, SEASON)

print(player_name)
display(playerdetails_df)
player_games_df

Remi Matthys


,playerGuid,playerName,season,playerdetailsTable
0,BVBL661847,Remi Matthys,2526,Amon Jeugd Gentson HSE C (BVBL1414HSE 3); Amo...


,playerGuid,playerName,gameGuid,gameDate,teamGuid,teamName,number,functionLetter,starter,totalMinutes,normalizedMinutes,freeThrows,fieldGoals,threePointers,totalScore,plusMinus,faults
0,BVBL661847,Remi Matthys,BVBL25261090OR00095407,2025-08-24T13:00:00.000Z,BVBL1414HSE 3,Amon Jeugd Gentson HSE C,12,S,True,18,20,2,2,0,4,10,5
1,BVBL661847,Remi Matthys,BVBL25261042OR00050608,2025-09-07T17:00:00.000Z,BVBL1414HSE 3,Amon Jeugd Gentson HSE C,12,S,False,18,20,1,8,3,12,-11,5
2,BVBL661847,Remi Matthys,BVBL25261446OR00501406,2025-08-30T20:20:00.000Z,BVBL1414HSE 3,Amon Jeugd Gentson HSE C,12,S,False,10,12,2,2,0,4,-13,2
3,BVBL661847,Remi Matthys,BVBL25261414OR00259504,2025-08-16T16:00:00.000Z,BVBL1414HSE 3,Amon Jeugd Gentson HSE C,12,S,False,22,24,9,10,6,25,28,2
4,BVBL661847,Remi Matthys,BVBL25269130OVHSE21ABD,2025-09-14T15:00:00.000Z,BVBL1414HSE 3,Amon Jeugd Gentson HSE C,12,S,False,20,22,3,6,0,9,-42,2
5,BVBL661847,Remi Matthys,BVBL25269130OVHSE21ADN,2025-09-20T16:00:00.000Z,BVBL1414HSE 3,Amon Jeugd Gentson HSE C,12,S,True,21,23,2,6,6,14,-15,2
6,BVBL661847,Remi Matthys,BVBL25269130OVHSE31BCK,2025-12-13T20:15:00.000Z,BVBL1414HSE 4,Amon Jeugd Gentson HSE D,13,S,False,15,17,2,4,0,6,-6,5
7,BVBL661847,Remi Matthys,BVBL25269130OVHSE21ADH,2026-03-14T16:00:00.000Z,BVBL1414HSE 3,Amon Jeugd Gentson HSE C,7,S,False,17,19,2,2,0,4,-3,2
8,BVBL661847,Remi Matthys,BVBL25269130OVHSE21ADB,2026-01-09T21:00:00.000Z,BVBL1414HSE 3,Amon Jeugd Gentson HSE C,6,S,False,18,18,0,2,0,2,-4,2
9,BVBL661847,Remi Matthys,BVBL25269130OVHSE21AED,2026-01-23T20:30:00.000Z,BVBL1414HSE 3,Amon Jeugd Gentson HSE C,9,S,True,29,33,4,12,3,19,-17,1


## 4. Competition-wide player stats (requires login)

The `playerStats` endpoint (`/api/competitions/{guid}/playerStats`) sits behind the site's Microsoft/Azure AD B2C login ("Stats Plus"), not a simple username/password form. Rather than handling your credentials in code, `login_and_get_context()` below opens a **real, visible browser window** and waits for you to log in yourself directly on Microsoft's hosted page. Once you're redirected back to the app, the resulting authenticated browser context (cookies + storage) is reused to call the API — your password never passes through this notebook.

**Why a background thread:** Jupyter's kernel (`ipykernel`) already runs its own asyncio event loop on the main thread. Playwright's sync API refuses to run inside a thread that has a running loop, and switching to Playwright's async API instead hits a *different* Windows-specific problem: `ipykernel` uses a `SelectorEventLoop`, but spawning the Playwright driver subprocess requires a `ProactorEventLoop` on Windows, so `await async_playwright().start()` fails with `NotImplementedError`. The fix that sidesteps both issues: run the plain **sync** Playwright API on a dedicated background thread, which has no asyncio loop of its own at all. A single-worker `ThreadPoolExecutor` keeps reusing that same thread across cells, since Playwright objects (browser/context) are only valid on the thread that created them.

In [13]:
# %pip install playwright
from concurrent.futures import ThreadPoolExecutor
from playwright.sync_api import sync_playwright

In [14]:
from concurrent.futures import ThreadPoolExecutor
from playwright.sync_api import sync_playwright

# All Playwright calls in this section run on this single background thread.
_pw_executor = ThreadPoolExecutor(max_workers=1)


def _login_sync(timeout_seconds):
    playwright = sync_playwright().start()
    browser = playwright.chromium.launch(headless=False)
    context = browser.new_context()
    page = context.new_page()
    page.goto(f"{BASE_URL}/signin")

    print("A browser window has opened.")
    print("Please log in with your Basketball Stats Vlaanderen account, then come back here.")
    print(f"Waiting up to {timeout_seconds}s for you to be redirected back to the app...")

    page.wait_for_url(f"{BASE_URL}/**", timeout=timeout_seconds * 1000)
    print("Login detected — session is ready.")

    return playwright, browser, context


def login_and_get_context(timeout_seconds=300):
    """Open a visible browser window for you to log in manually, then return
    (playwright, browser, context) with an authenticated session.

    Runs on the dedicated Playwright thread (see markdown above). Keep the returned
    objects alive for as long as you want to keep making authenticated requests; call
    close_playwright(playwright, browser) when you're done.
    """
    return _pw_executor.submit(_login_sync, timeout_seconds).result()

In [15]:
 sync_playwright().start()

Error: It looks like you are using Playwright Sync API inside the asyncio loop.
Please use the Async API instead.

In [10]:
login_and_get_context(timeout_seconds=300)

NotImplementedError: 

In [67]:
def _fetch_all_player_stats_sync(context, competition_guid, page_size, max_pages):
    all_rows = []
    page_num = 1

    while page_num <= max_pages:
        resp = context.request.get(
            f"{BASE_URL}/api/competitions/{competition_guid}/playerStats",
            params={"page": page_num, "pageSize": page_size},
        )
        if not resp.ok:
            raise ValueError(f"Request failed ({resp.status}): {resp.text()}")

        data = resp.json()

        if isinstance(data, list):
            # No pagination wrapper -- assume the endpoint already returns everything.
            all_rows.extend(data)
            break

        rows = data.get("data") or data.get("rows") or data.get("items") or []
        if not rows:
            break

        all_rows.extend(rows)

        if len(rows) < page_size:
            break
        page_num += 1

    return pd.DataFrame(all_rows)


def fetch_all_player_stats(context, competition_guid, page_size=100, max_pages=200):
    """Paginate through /api/competitions/{guid}/playerStats using an authenticated
    Playwright context (from login_and_get_context) and return all records as a DataFrame.

    Runs on the dedicated Playwright thread (see markdown above) since `context` is only
    valid on the thread that created it.

    The exact pagination convention of the endpoint hasn't been confirmed (it 401s without
    login), so this tries the common `page`/`pageSize` query params first; if the response
    is a plain list with no pagination wrapper, it is returned as-is after a single call.
    Inspect the raw response once (see the debug cell below) and adjust this if needed.
    """
    return _pw_executor.submit(
        _fetch_all_player_stats_sync, context, competition_guid, page_size, max_pages
    ).result()

### Log in once, then fetch both competitions

Run the cell below and complete the login in the browser window that opens. This only needs to happen once per notebook session — reuse `context` for as many calls as you like afterwards.

In [68]:
playwright, browser, context = login_and_get_context()

NotImplementedError: 

In [59]:
# Debug: inspect the raw shape of one page before trusting fetch_all_player_stats' parsing above.
def _debug_call():
    resp = context.request.get(
        f"{BASE_URL}/api/competitions/BVBL25269130OVHSE31A/playerStats",
        params={"page": 1, "pageSize": 5},
    )
    return resp.status, resp.json()


debug_status, debug_body = _pw_executor.submit(_debug_call).result()
print(debug_status)
debug_body

NameError: name 'context' is not defined

In [58]:
COMPETITION_GUID_A = "BVBL25269130OVHSE31A"
COMPETITION_GUID_B = "BVBL25269130OVHSE31B"

player_stats_a = fetch_all_player_stats(context, COMPETITION_GUID_A)
player_stats_b = fetch_all_player_stats(context, COMPETITION_GUID_B)

print(player_stats_a.shape, player_stats_b.shape)
all_player_stats = pd.concat(
    [player_stats_a.assign(competitionGuid=COMPETITION_GUID_A),
     player_stats_b.assign(competitionGuid=COMPETITION_GUID_B)],
    ignore_index=True,
)
all_player_stats

NameError: name 'context' is not defined

### Add team metadata (`playerdetailsTable`)

Once we've seen the real column names from the debug cell above, build a `playerdetailsTable` column the same way as in section 3 — compactly listing which team(s) each player belongs to. Adjust the column names below (`teamNameCol`/`teamGuidCol`/`playerGuidCol`) to match what the debug output actually shows.

In [ ]:
# Adjust these to match the real field names once you've seen the debug output above.
player_guid_col = "Guid"
team_name_col = "TeamName"
team_guid_col = "TeamGuid"

playerdetails_all_df = (
    all_player_stats[[player_guid_col, team_name_col, team_guid_col]]
    .drop_duplicates()
    .groupby(player_guid_col)
    .apply(lambda g: "; ".join(f"{r[team_name_col]} ({r[team_guid_col]})" for _, r in g.iterrows()))
    .reset_index(name="playerdetailsTable")
)
playerdetails_all_df

In [ ]:
def _close_sync(playwright, browser):
    browser.close()
    playwright.stop()


def close_playwright(playwright, browser):
    """Close the browser and stop Playwright (must run on the same background thread)."""
    return _pw_executor.submit(_close_sync, playwright, browser).result()


# Once you're done making authenticated calls, close the browser/session.
close_playwright(playwright, browser)
_pw_executor.shutdown()